# PitchOps — Football MLOps trên Colab

Notebook của **Nguyễn Hồng Quang**. Chạy CPU, không cần GPU. Chọn **Sao chép vào Drive** trước khi sửa. Tất cả cài đặt, code, kiểm thử và training trong quy trình này chạy ở Colab.

**Đã có:** dữ liệu trận 5 giải lớn, dự báo bàn/thắng–hòa–thua, góc/cú sút thử nghiệm, Monte Carlo quốc nội, dashboard HTML tự chứa, metrics/lineage/cache. **Chưa có:** coverage mọi cầu thủ/quốc gia, live service, dự báo heatmap/rating/MOTM, engine cúp và scheduler khi Colab đóng.

Dữ liệu football-data.co.uk chỉ dùng cá nhân/học thuật theo điều khoản nguồn, tải tự động nhẹ tối đa hai lần/tuần. Giữ cache/ledger qua các phiên hoặc nhập file thủ công. Không chia sẻ raw data/model khi chưa có quyền. [Code & hướng dẫn](https://github.com/HongQuang26/pitchops).


## 1. Lấy code và cài thư viện

Clone lần đầu, lần sau tái dùng checkout. Không tự ghi đè các thay đổi bạn đang sửa. In commit để đối chiếu code chạy thật. Môi trường chính xác được ghi lại trong artifacts sau khi chạy.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO = Path('/content/pitchops')
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/HongQuang26/pitchops.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
print('Code commit:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())
print('Ready:', REPO)


## 2. Nơi giữ cache và checkpoint

Đặt `USE_DRIVE=True` nếu muốn dữ liệu/model còn sau khi runtime bị thu hồi. Bạn tự xác nhận quyền Google nếu được hỏi. Mặc định chỉ dùng `/content`; hãy tải backup và khôi phục cache trước lần chạy sau nếu không gắn Drive. Không reset ledger để vượt quota hoặc chạy nhiều phiên đồng thời.

In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
WORKSPACE = Path('/content/drive/MyDrive/PitchOps/workspace') if USE_DRIVE else Path('/content/pitchops_workspace')
WORKSPACE.mkdir(parents=True, exist_ok=True)
print('Workspace:', WORKSPACE)
print('Persistent across runtimes:', USE_DRIVE)


## 3. Kiểm thử các hợp đồng quan trọng

Kiểm tra rò rỉ dữ liệu tương lai, xử lý trận cùng ngày, tổng xác suất, dữ liệu trùng/âm, coverage mùa giải, nhãn cầu thủ và cache budget. Không huấn luyện ở GitHub Actions hoặc máy khác.

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=REPO, check=True)


## 4. Thu thập → features → train → evaluate → simulate

Mặc định 2 workbook cho mùa 2025/26 và 2026/27, học 5 giải. Lần chạy lại dùng cache. Có thể chọn chỉ `LEAGUES=['E0']` để thử nhanh. `AS_OF` loại mọi trận từ ngày đó trở đi; `None` dùng kết quả trước hôm nay. Không tự điền dữ liệu giả nếu nguồn lỗi.

Mô hình được chọn bằng validation phía trước, đánh giá trên test phía sau rồi refit. Log loss/Brier thấp hơn tốt hơn. Nếu candidate không vượt baseline, hệ thống dùng baseline. Thống kê góc/cú sút đang thử nghiệm; chưa hiệu chỉnh calibration. Mô phỏng mùa giữ sức mạnh cố định, cặp đấu còn lại suy ra theo vòng tròn hai lượt và chưa mô hình hóa mọi luật đối đầu/playoff.

In [ ]:
from pitchops.pipeline import run
SEASONS = ['2526', '2627']
LEAGUES = ['E0', 'SP1', 'D1', 'I1', 'F1']
AS_OF = None
LOCAL_FILES = None  # Ví dụ {'2526':'/content/manual-2526.xlsx', '2627':'/content/manual-2627.xlsx'}
payload, run_dir = run(WORKSPACE, seasons=SEASONS, leagues=LEAGUES, as_of=AS_OF, simulations=5000, local_files=LOCAL_FILES)
print('Artifacts:', run_dir)


## 5. Dashboard phong cách sân cỏ

Chạy ô này sau mỗi lần tạo snapshot. Dashboard lấy dự đoán từ mô hình vừa huấn luyện, có lọc giải, chọn hai đội, đổi sân, phân phối tỉ số, bảng đua và model/data scorecard. Không có CDN, tunnel công khai hay server ngoài Colab. Cuộn trong khung để xem toàn bộ.

In [ ]:
from IPython.display import HTML, display
import html
page = (run_dir / 'dashboard.html').read_text(encoding='utf-8')
display(HTML('<iframe title="PitchOps dashboard" sandbox="allow-scripts allow-downloads" style="width:100%;height:1050px;border:0;border-radius:16px" srcdoc="'+html.escape(page, quote=True)+'"></iframe>'))


## 6. Xem metrics và dấu vết dữ liệu

Trọng số của bản refit được lưu `model.joblib`; chỉ load joblib từ artifact tin cậy. `requirements-lock.txt` ghi lại môi trường vừa chạy. `snapshot.json` chứa các dự báo; `model_card.json` có thời điểm, hash dữ liệu và commit code.

In [ ]:
import json
print(json.dumps({'meta': payload['meta'], 'metrics': payload['metrics'], 'quality': payload['quality']}, ensure_ascii=False, indent=2))


## 7. MLflow tùy chọn

Bật `ENABLE_MLFLOW=True` để ghi experiment tại workspace. Không cần server MLflow hoặc tài khoản. Đây là run tracking; chưa có production registry/promotion tự động.

In [ ]:
ENABLE_MLFLOW = False
if ENABLE_MLFLOW:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[tracking]'], check=True)
    from pitchops.pipeline import track_mlflow
    track_mlflow(payload, run_dir)
    print('MLflow run logged')
else:
    print('Optional MLflow skipped; core JSON metrics and lineage are already saved.')


## 8. Cầu thủ và heatmap tùy chọn

Scorer projection cần CSV thật: `player_id,player,team,date,minutes,goals`. Đặt ngày dự báo và số phút dự kiến trước trận; không dùng số phút thực tế của trận mục tiêu. XI lịch sử cần thêm `rating,position` (GK/DF/MF/FW). Các ví dụ dưới đây bị tắt đến khi bạn cung cấp dữ liệu, không tạo cầu thủ giả.

In [ ]:
PLAYER_CSV = None
if PLAYER_CSV:
    import pandas as pd
    from pitchops.players import scorer_projection, select_form_xi
    player_logs = pd.read_csv(PLAYER_CSV)
    EXPECTED_MINUTES = {}  # player_id thật -> số phút dự kiến
    print(scorer_projection(player_logs, payload['meta']['forecast_date'], EXPECTED_MINUTES))
    # display(pd.DataFrame(select_form_xi(player_logs, payload['meta']['forecast_date'], period='week')))
else:
    print('Player models need verified player match logs; no synthetic player outputs.')
# StatsBomb adapter, chỉ bật sau khi đọc license và dùng IDs từ competitions/matches:
# from pitchops.events import StatsBomb
# sb = StatsBomb(WORKSPACE / 'data' / 'statsbomb')
# display(sb.competitions())
# locations = sb.player_locations(match_id=..., player_id=...)
# Đây là tọa độ sự kiện quá khứ, không phải tracking/predicted heatmap.


## 9. Tải dashboard hoặc backup

Bật cờ bên dưới rồi chạy. Backup gồm cache/ledger và artifacts, không phải secret. Khôi phục cùng cache trước khi tải tự động ở runtime mới. Không đẩy backup này vào repo public. Chỉ chia sẻ output nếu quyền dữ liệu của bạn cho phép.

In [ ]:
DOWNLOAD_DASHBOARD = False
DOWNLOAD_BACKUP = False
if DOWNLOAD_DASHBOARD or DOWNLOAD_BACKUP:
    from google.colab import files
if DOWNLOAD_DASHBOARD:
    files.download(str(run_dir / 'dashboard.html'))
if DOWNLOAD_BACKUP:
    import shutil
    backup = shutil.make_archive('/content/PitchOps_private_backup', 'zip', WORKSPACE)
    files.download(backup)


## 10. Sửa code và commit/push từ Colab

Sửa file trong `/content/pitchops/src` hoặc dùng `%%writefile`. Muốn push, thêm `GITHUB_TOKEN` vào Colab Secrets (fine-grained, chỉ repo này, Contents write), tự cho phép notebook truy cập, rồi đặt `PUSH_CODE=True`. Token không in ra, không nằm trong Git URL hoặc file. Chỉ stage danh sách file code; không stage dataset/model/output. Ô này tắt mặc định.

In [ ]:
PUSH_CODE = False
if PUSH_CODE:
    from google.colab import userdata
    import tempfile
    token = userdata.get('GITHUB_TOKEN')
    env = os.environ.copy()
    env['PITCHOPS_GH_TOKEN'] = token
    env['GIT_TERMINAL_PROMPT'] = '0'
    with tempfile.TemporaryDirectory() as temp:
        askpass = Path(temp) / 'askpass.py'
        askpass.write_text('#!/usr/bin/env python3\nimport os,sys\nprint("x-access-token" if "Username" in sys.argv[1] else os.environ["PITCHOPS_GH_TOKEN"])\n')
        askpass.chmod(0o700)
        env['GIT_ASKPASS'] = str(askpass)
        subprocess.run(['git','config','user.name','Nguyen Hong Quang'], check=True)
        subprocess.run(['git','config','user.email','144700005+HongQuang26@users.noreply.github.com'], check=True)
        subprocess.run(['git','add','src','tests','docs','notebooks','README.md','LICENSE','pyproject.toml','.gitignore'], check=True)
        print(subprocess.check_output(['git','diff','--cached','--stat'], text=True))
        if subprocess.run(['git','diff','--cached','--quiet']).returncode:
            subprocess.run(['git','commit','-m','Update PitchOps from Colab'], check=True)
            subprocess.run(['git','push','origin','main'], env=env, check=True)
        else:
            print('No staged code changes.')
    env.pop('PITCHOPS_GH_TOKEN', None)
    del token
else:
    print('GitHub write skipped. Use PUSH_CODE=True only for intentional code updates.')


## Tuần sau và các bước tiếp

Mở lại notebook, nối runtime, khôi phục workspace/cache, chạy lại pipeline. Không có job chạy nền sau khi Colab tắt. Muốn tự động hóa không cần mở Colab phải bổ sung scheduler/compute ổn định theo quyết định sau này.

Lộ trình: nhiều mùa và rolling backtests → calibration riêng → provider player/event data → expected lineup/minutes → scorer/rating/MOTM → spatial models → engine cúp quốc tế → production registry/scheduler/API. Xem `docs/ARCHITECTURE.md` để biết phần nào đã có và phần nào chưa.
